# Prediksi Jumlah Perjalanan Wisatawan Nusantara Berdasarkan Kabupaten/Kota
# di Provinsi Sulawesi Tengah Menggunakan Metode Regresi Linear

**Nama Proyek:** Analisis & Prediksi Wisatawan Nusantara Sulawesi Tengah  
**Dataset:** Data BPS Provinsi Sulawesi Tengah (2024–2026)  
**Metode:** Regresi Linear Sederhana & Berganda  
**Tools:** Python 3.x, scikit-learn, pandas, matplotlib, seaborn, statsmodels

---
## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Pengaturan tampilan
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.0f}'.format)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')
PALETTE = sns.color_palette('Set2', 14)

print('✅ Semua library berhasil diimpor!')

---
## 2. Load Data

In [ ]:
# Baca data dari baris ke-5 (header index 3 = row ke-4, data mulai row ke-5)
df_raw = pd.read_excel('Gabungan_Data_Wisatawan_Sulteng_2024_2026.xlsx', header=None)

# Ambil baris header (baris ke-4, index 3) dan data (baris 5 dst, index 4+)
header_row = df_raw.iloc[3].tolist()
df = df_raw.iloc[4:].copy()
df.columns = header_row
df = df.reset_index(drop=True)

# Rename kolom
df.columns = ['Tahun', 'Kabupaten_Kota', 'Januari', 'Februari', 'Maret', 'April',
              'Mei', 'Juni', 'Juli', 'Agustus', 'September', 'Oktober', 'November', 'Desember', 'Tahunan']

# Ganti '-' dengan NaN
df = df.replace('-', np.nan)

# Konversi tipe data
bulan_cols = ['Januari','Februari','Maret','April','Mei','Juni',
              'Juli','Agustus','September','Oktober','November','Desember','Tahunan']
for col in bulan_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['Tahun'] = df['Tahun'].astype(int)

print('Shape data mentah:', df.shape)
print('Tahun tersedia:', sorted(df['Tahun'].unique()))
df.head(10)

In [ ]:
# Pisahkan data total provinsi dan data per kabupaten/kota
df_prov = df[df['Kabupaten_Kota'] == 'Sulawesi Tengah'].copy().reset_index(drop=True)
df_kab  = df[df['Kabupaten_Kota'] != 'Sulawesi Tengah'].copy().reset_index(drop=True)

# Data 2024 & 2025 lengkap (12 bulan + Tahunan)
df_lengkap = df_kab[df_kab['Tahun'].isin([2024, 2025])].copy()
# Data 2026 hanya Jan–Apr
df_2026 = df_kab[df_kab['Tahun'] == 2026].copy()

print(f'Kabupaten/Kota unik: {df_kab["Kabupaten_Kota"].nunique()}')
print(f'Data lengkap (2024–2025): {df_lengkap.shape[0]} baris')
print(f'Data 2026 (parsial): {df_2026.shape[0]} baris')
print('\nDaftar Kabupaten/Kota:')
print(df_kab['Kabupaten_Kota'].unique())

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('=== INFORMASI DATASET ===')
df_lengkap.info()
print('\n=== STATISTIK DESKRIPTIF (Data Tahunan per Kabupaten/Kota) ===')
df_lengkap[['Tahun','Tahunan']].groupby('Tahun').describe()

In [ ]:
print('=== MISSING VALUES ===')
print(df_kab.isnull().sum())
print('\n=== JUMLAH WISATAWAN TOTAL PER TAHUN (Sulawesi Tengah) ===')
print(df_prov[['Tahun','Tahunan']].to_string(index=False))

In [ ]:
# ---- GRAFIK 1: Total Wisatawan Provinsi per Tahun ----
fig, ax = plt.subplots(figsize=(9, 5))
tahun_prov = df_prov['Tahun'].values
total_prov = df_prov['Tahunan'].values

bars = ax.bar(tahun_prov.astype(str), total_prov, color=['#2196F3','#4CAF50','#FF9800'], width=0.5, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, total_prov):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100000,
                f'{val:,.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Total Wisatawan Nusantara Provinsi Sulawesi Tengah\nper Tahun (2024–2026)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Tahun', fontsize=12)
ax.set_ylabel('Jumlah Wisatawan', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.1f}M'))
ax.set_ylim(0, max(total_prov[~np.isnan(total_prov)]) * 1.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('grafik/01_total_wisatawan_provinsi.png', bbox_inches='tight')
plt.show()
print('✅ Grafik 1 tersimpan')

In [ ]:
# ---- GRAFIK 2: Perbandingan Wisatawan per Kabupaten/Kota (2024 vs 2025) ----
df_24 = df_lengkap[df_lengkap['Tahun']==2024][['Kabupaten_Kota','Tahunan']].set_index('Kabupaten_Kota')
df_25 = df_lengkap[df_lengkap['Tahun']==2025][['Kabupaten_Kota','Tahunan']].set_index('Kabupaten_Kota')
df_compare = df_24.join(df_25, lsuffix='_2024', rsuffix='_2025').sort_values('Tahunan_2025', ascending=True)

fig, ax = plt.subplots(figsize=(11, 7))
y = np.arange(len(df_compare))
width = 0.35
b1 = ax.barh(y - width/2, df_compare['Tahunan_2024'], width, label='2024', color='#42A5F5', edgecolor='white')
b2 = ax.barh(y + width/2, df_compare['Tahunan_2025'], width, label='2025', color='#66BB6A', edgecolor='white')

ax.set_yticks(y)
ax.set_yticklabels(df_compare.index, fontsize=10)
ax.set_xlabel('Jumlah Wisatawan', fontsize=12)
ax.set_title('Perbandingan Jumlah Wisatawan Nusantara\nper Kabupaten/Kota di Sulawesi Tengah (2024 vs 2025)',
             fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('grafik/02_perbandingan_kabkota_2024_2025.png', bbox_inches='tight')
plt.show()
print('✅ Grafik 2 tersimpan')

In [ ]:
# ---- GRAFIK 3: Tren Bulanan Provinsi Sulawesi Tengah (2024 vs 2025) ----
bulan_labels = ['Jan','Feb','Mar','Apr','Mei','Jun','Jul','Agu','Sep','Okt','Nov','Des']
bulan_cols12 = ['Januari','Februari','Maret','April','Mei','Juni','Juli','Agustus','September','Oktober','November','Desember']

prov_24 = df_prov[df_prov['Tahun']==2024][bulan_cols12].values.flatten()
prov_25 = df_prov[df_prov['Tahun']==2025][bulan_cols12].values.flatten()
prov_26_raw = df_prov[df_prov['Tahun']==2026][['Januari','Februari','Maret','April']].values.flatten()

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(12)
ax.plot(x, prov_24, 'o-', color='#2196F3', linewidth=2.5, markersize=7, label='2024')
ax.plot(x, prov_25, 's-', color='#4CAF50', linewidth=2.5, markersize=7, label='2025')
ax.plot(np.arange(4), prov_26_raw, '^--', color='#FF9800', linewidth=2.5, markersize=8, label='2026 (Jan–Apr)')

ax.set_xticks(x)
ax.set_xticklabels(bulan_labels, fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v/1000:.0f}K'))
ax.set_xlabel('Bulan', fontsize=12)
ax.set_ylabel('Jumlah Wisatawan', fontsize=12)
ax.set_title('Tren Bulanan Wisatawan Nusantara Sulawesi Tengah (2024–2026)',
             fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('grafik/03_tren_bulanan_provinsi.png', bbox_inches='tight')
plt.show()
print('✅ Grafik 3 tersimpan')

In [ ]:
# ---- GRAFIK 4: Heatmap Korelasi Bulanan ----
df_corr = df_lengkap[bulan_cols12 + ['Tahunan']].copy()
corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax,
            xticklabels=['Jan','Feb','Mar','Apr','Mei','Jun','Jul','Agu','Sep','Okt','Nov','Des','Total'],
            yticklabels=['Jan','Feb','Mar','Apr','Mei','Jun','Jul','Agu','Sep','Okt','Nov','Des','Total'])
ax.set_title('Heatmap Korelasi Antar Bulan terhadap Total Wisatawan Tahunan',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('grafik/04_heatmap_korelasi.png', bbox_inches='tight')
plt.show()
print('✅ Grafik 4 tersimpan')

In [ ]:
# ---- GRAFIK 5: Distribusi Total Tahunan per Kabupaten/Kota ----
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df_lengkap, x='Kabupaten_Kota', y='Tahunan', palette='Set2', ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
ax.set_xlabel('Kabupaten/Kota', fontsize=11)
ax.set_ylabel('Jumlah Wisatawan Tahunan', fontsize=11)
ax.set_title('Distribusi Wisatawan Tahunan per Kabupaten/Kota (2024–2025)',
             fontsize=13, fontweight='bold', pad=15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('grafik/05_distribusi_tahunan_kabkota.png', bbox_inches='tight')
plt.show()
print('✅ Grafik 5 tersimpan')

---
## 4. Preprocessing Data

In [ ]:
# Encode Kabupaten/Kota menjadi angka
le = LabelEncoder()
df_model = df_lengkap.copy()
df_model['Kab_Encoded'] = le.fit_transform(df_model['Kabupaten_Kota'])

# Buat fitur: Tahun dan Kab_Encoded sebagai X, Tahunan sebagai y
X = df_model[['Tahun', 'Kab_Encoded']].values
y = df_model['Tahunan'].values

# Cek NaN
mask = ~np.isnan(y)
X = X[mask]
y = y[mask]

print(f'Jumlah sampel setelah preprocessing: {len(y)}')
print(f'Fitur: Tahun, Kab_Encoded')
print(f'Target: Tahunan (jumlah wisatawan)')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'\nData latih: {len(X_train)} sampel')
print(f'Data uji  : {len(X_test)} sampel')

---
## 5. Regresi Linear — Model 1: Sederhana (Tahun → Total Provinsi)

In [ ]:
# Model 1: Regresi Linear Sederhana — Tahun vs Total Wisatawan Provinsi
df_prov_lengkap = df_prov[df_prov['Tahun'].isin([2024,2025])].dropna(subset=['Tahunan']).copy()
X1 = df_prov_lengkap[['Tahun']].values
y1 = df_prov_lengkap['Tahunan'].values

model1 = LinearRegression()
model1.fit(X1, y1)

print('=== MODEL 1: Regresi Linear Sederhana (Tahun → Total Wisatawan Provinsi) ===')
print(f'Intercept (β₀)   : {model1.intercept_:,.2f}')
print(f'Koefisien (β₁)   : {model1.coef_[0]:,.2f}')
print(f'\nPersamaan: Ŷ = {model1.intercept_:,.0f} + {model1.coef_[0]:,.0f} × Tahun')

# Prediksi 2024, 2025, dan 2026
tahun_pred = np.array([[2024],[2025],[2026]])
y_pred1 = model1.predict(tahun_pred)
print('\nPrediksi:')
for t, p in zip([2024,2025,2026], y_pred1):
    print(f'  Tahun {t}: {p:,.0f} wisatawan')

---
## 6. Regresi Linear — Model 2: Berganda (Kabupaten/Kota)

In [ ]:
# Model 2: Regresi Linear Berganda — Fitur bulan Q1 (Jan–Apr) untuk prediksi total tahunan
# Relevan karena data 2026 hanya tersedia Jan–Apr
bulan_q1 = ['Januari','Februari','Maret','April']
df_fit = df_lengkap.dropna(subset=['Tahunan'] + bulan_q1).copy()

X2 = df_fit[bulan_q1].values
y2 = df_fit['Tahunan'].values

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

model2 = LinearRegression()
model2.fit(X2_train, y2_train)

y2_pred_test = model2.predict(X2_test)
y2_pred_train = model2.predict(X2_train)

r2  = r2_score(y2_test, y2_pred_test)
mae = mean_absolute_error(y2_test, y2_pred_test)
rmse = np.sqrt(mean_squared_error(y2_test, y2_pred_test))

print('=== MODEL 2: Regresi Linear Berganda (Jan+Feb+Mar+Apr → Total Tahunan) ===')
print(f'Intercept (β₀) : {model2.intercept_:,.2f}')
for i, bulan in enumerate(bulan_q1):
    print(f'Koefisien {bulan:>8}: {model2.coef_[i]:,.4f}')
print(f'\n=== EVALUASI MODEL (Test Set) ===')
print(f'R²   : {r2:.4f}  ({r2*100:.2f}%)')
print(f'MAE  : {mae:,.0f}')
print(f'RMSE : {rmse:,.0f}')

In [ ]:
# Statsmodels OLS untuk detail statistik
X2_sm = sm.add_constant(X2)
ols_model = sm.OLS(y2, X2_sm).fit()
print(ols_model.summary())

---
## 7. Prediksi Total Tahunan 2026 per Kabupaten/Kota

In [ ]:
# Prediksi 2026 menggunakan data Jan–Apr yang tersedia
df_2026_pred = df_2026[['Kabupaten_Kota'] + bulan_q1].copy()
df_2026_pred = df_2026_pred.dropna(subset=bulan_q1)

X_2026 = df_2026_pred[bulan_q1].values
y_2026_pred = model2.predict(X_2026)

df_2026_pred['Prediksi_Tahunan'] = y_2026_pred

# Gabungkan dengan data 2024 dan 2025
df_summary = pd.merge(
    df_lengkap[df_lengkap['Tahun']==2024][['Kabupaten_Kota','Tahunan']].rename(columns={'Tahunan':'2024'}),
    df_lengkap[df_lengkap['Tahun']==2025][['Kabupaten_Kota','Tahunan']].rename(columns={'Tahunan':'2025'}),
    on='Kabupaten_Kota'
)
df_summary = pd.merge(df_summary,
    df_2026_pred[['Kabupaten_Kota','Prediksi_Tahunan']].rename(columns={'Prediksi_Tahunan':'Prediksi_2026'}),
    on='Kabupaten_Kota'
)
df_summary['Pertumbuhan_2024_2025(%)'] = ((df_summary['2025'] - df_summary['2024']) / df_summary['2024'] * 100).round(2)
df_summary['Pertumbuhan_2025_2026(%)'] = ((df_summary['Prediksi_2026'] - df_summary['2025']) / df_summary['2025'] * 100).round(2)

print('=== TABEL PREDIKSI WISATAWAN 2026 PER KABUPATEN/KOTA ===')
pd.set_option('display.float_format', '{:,.0f}'.format)
df_summary.sort_values('Prediksi_2026', ascending=False)

---
## 8. Visualisasi Hasil Model

In [ ]:
# ---- GRAFIK 6: Aktual vs Prediksi (Model 2 Test Set) ----
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y2_test, y2_pred_test, color='#2196F3', alpha=0.7, s=80, edgecolors='white', linewidth=1, zorder=3)
min_val, max_val = min(y2_test.min(), y2_pred_test.min()), max(y2_test.max(), y2_pred_test.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Garis Ideal (y=x)')
ax.set_xlabel('Nilai Aktual (Wisatawan)', fontsize=12)
ax.set_ylabel('Nilai Prediksi (Wisatawan)', fontsize=12)
ax.set_title(f'Aktual vs Prediksi — Model Regresi Linear\nR² = {r2:.4f} | MAE = {mae:,.0f} | RMSE = {rmse:,.0f}',
             fontsize=12, fontweight='bold', pad=15)
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('grafik/06_aktual_vs_prediksi.png', bbox_inches='tight')
plt.show()
print('✅ Grafik 6 tersimpan')

In [ ]:
# ---- GRAFIK 7: Prediksi 2026 vs Aktual 2024 & 2025 ----
df_plot = df_summary.sort_values('Prediksi_2026', ascending=True)
kabkota = df_plot['Kabupaten_Kota'].values
y_pos = np.arange(len(kabkota))

fig, ax = plt.subplots(figsize=(11, 7))
w = 0.28
ax.barh(y_pos - w, df_plot['2024'], w, label='2024 (Aktual)', color='#42A5F5', edgecolor='white')
ax.barh(y_pos,     df_plot['2025'], w, label='2025 (Aktual)', color='#66BB6A', edgecolor='white')
ax.barh(y_pos + w, df_plot['Prediksi_2026'], w, label='2026 (Prediksi)', color='#FFA726', edgecolor='white', hatch='//')

ax.set_yticks(y_pos)
ax.set_yticklabels(kabkota, fontsize=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
ax.set_xlabel('Jumlah Wisatawan', fontsize=12)
ax.set_title('Perbandingan Aktual 2024–2025 dan Prediksi 2026\nJumlah Wisatawan Nusantara per Kabupaten/Kota Sulawesi Tengah',
             fontsize=12, fontweight='bold', pad=15)
ax.legend(fontsize=10, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('grafik/07_prediksi_2026_vs_aktual.png', bbox_inches='tight')
plt.show()
print('✅ Grafik 7 tersimpan')

In [ ]:
# ---- GRAFIK 8: Residual Plot ----
residuals = y2_test - y2_pred_test

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Residual vs Fitted
axes[0].scatter(y2_pred_test, residuals, color='#7E57C2', alpha=0.7, s=70, edgecolors='white')
axes[0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Nilai Prediksi', fontsize=11)
axes[0].set_ylabel('Residual', fontsize=11)
axes[0].set_title('Residual vs Nilai Prediksi', fontsize=12, fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Distribusi Residual
axes[1].hist(residuals, bins=8, color='#7E57C2', alpha=0.8, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual', fontsize=11)
axes[1].set_ylabel('Frekuensi', fontsize=11)
axes[1].set_title('Distribusi Residual', fontsize=12, fontweight='bold')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Analisis Residual Model Regresi Linear', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('grafik/08_analisis_residual.png', bbox_inches='tight')
plt.show()
print('✅ Grafik 8 tersimpan')

---
## 9. Evaluasi Model — Ringkasan

In [ ]:
# MAPE
mape = np.mean(np.abs((y2_test - y2_pred_test) / y2_test)) * 100

print('='*55)
print('  RINGKASAN EVALUASI MODEL REGRESI LINEAR')
print('='*55)
print(f'  R² (Koefisien Determinasi) : {r2:.4f}  ({r2*100:.2f}%)')
print(f'  MAE  (Mean Abs. Error)     : {mae:>12,.0f}')
print(f'  RMSE (Root Mean Sq. Error) : {rmse:>12,.0f}')
print(f'  MAPE (Mean Abs. Pct Err)   : {mape:>11.2f}%')
print('='*55)

if r2 >= 0.9:
    kualitas = 'Sangat Baik (≥ 90%)'
elif r2 >= 0.7:
    kualitas = 'Baik (70–89%)'
elif r2 >= 0.5:
    kualitas = 'Cukup (50–69%)'
else:
    kualitas = 'Lemah (< 50%)'
print(f'  Kualitas Model             : {kualitas}')
print('='*55)

---
## 10. Interpretasi & Kesimpulan

### Interpretasi Hasil

**1. Tren Wisatawan 2024–2025:**  
Jumlah wisatawan nusantara di Sulawesi Tengah menunjukkan pertumbuhan positif dari **9.329.221** (2024) menjadi **11.668.933** (2025), atau naik sekitar **25,1%**. Kota Palu konsisten menjadi destinasi unggulan dengan kontribusi terbesar, diikuti Donggala dan Morowali.

**2. Kinerja Model Regresi Linear Berganda:**  
Model menggunakan data bulan Januari–April sebagai prediktor total tahunan. Nilai R² yang diperoleh menunjukkan seberapa besar variasi data dapat dijelaskan oleh model. Koefisien setiap bulan menunjukkan kontribusi masing-masing bulan terhadap total tahunan.

**3. Prediksi 2026:**  
Berdasarkan data Jan–Apr 2026, model memprediksi total wisatawan 2026 untuk setiap kabupaten/kota. Kota Palu kembali diprediksi tertinggi, sementara Banggai Laut diprediksi terendah.

**4. Pola Musiman:**  
Bulan April cenderung menjadi puncak kunjungan, sementara Agustus dan September relatif rendah. Pola ini konsisten di 2024 dan 2025.

### Kesimpulan
- Model Regresi Linear berhasil memodelkan hubungan antara data bulanan awal tahun dengan total wisatawan tahunan.
- Data 4 bulan pertama (Jan–Apr) sudah cukup representatif untuk memproyeksikan total tahunan dengan akurasi yang memadai.
- Kota Palu, Donggala, dan Morowali merupakan 3 tujuan wisata terbesar di Sulawesi Tengah.

### Rekomendasi Praktis
1. **Fokus pengembangan infrastruktur** di Kota Palu dan Donggala yang menjadi destinasi utama.
2. **Strategi promosi bulan sepi** (Agustus–September) untuk meratakan distribusi kunjungan sepanjang tahun.
3. **Pemantauan data triwulanan** sudah cukup untuk proyeksi tahunan, memudahkan perencanaan dinas pariwisata.
4. **Dorong pertumbuhan kabupaten kecil** seperti Buol dan Banggai Laut yang masih memiliki potensi wisata yang belum tergali.
5. **Update model** secara berkala saat data baru tersedia untuk meningkatkan akurasi prediksi.